Ran on Google colab T4 GPU

In [ ]:
!pip install transformers==4.40.0 accelerate==0.29.3 -q

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, hamming_loss

DISTORTION_LABELS = [
    "All-or-nothing thinking",
    "Overgeneralization",
    "Mental filter",
    "Should statements",
    "Labeling",
    "Personalization",
    "Catastrophising/Magnification",
    "Emotional Reasoning",
    "Mind Reading",
    "Fortune-telling",
]
NUM_LABELS       = len(DISTORTION_LABELS)   # 10
BASE_MODEL       = "distilbert-base-uncased"
MAX_LEN          = 128
BATCH_SIZE       = 16
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 10   
WARMUP_STEPS     = 200  
WEIGHT_DECAY     = 0.01
THRESHOLD        = 0.2  
POS_WEIGHT_SCALE = 3.0  
SEED             = 42
MODEL_DIR        = "distilbert_cognitive_distortion"

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Threshold     : {THRESHOLD}")
print(f"PosWeightScale: {POS_WEIGHT_SCALE}")


In [ ]:
train_df      = pd.read_csv("train.csv")
test_df       = pd.read_csv("test.csv")
class_weights = np.load("class_weights.npy")

print(f"Train : {len(train_df)} rows")
print(f"Test  : {len(test_df)} rows")
print(f"Class weights : {class_weights.round(4)}")

missing = [l for l in DISTORTION_LABELS if l not in train_df.columns]
assert not missing, f"Missing label columns: {missing}"
print("All label columns verified.")

In [ ]:
class CognitiveDistortionDataset(Dataset):
    """
    PyTorch Dataset for multi-label cognitive distortion classification.
    Each sample returns input_ids, attention_mask, and a 10-dim float label vector.
    """
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.texts     = df["text"].astype(str).tolist()
        self.labels    = df[DISTORTION_LABELS].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),       
            "attention_mask": enc["attention_mask"].squeeze(0),  
            "labels"        : torch.tensor(self.labels[idx], dtype=torch.float32)
        }

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_MODEL)

train_dataset = CognitiveDistortionDataset(train_df, tokenizer, MAX_LEN)
test_dataset  = CognitiveDistortionDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Test  batches : {len(test_loader)}")

batch = next(iter(train_loader))
print(f"input_ids shape     : {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape        : {batch['labels'].shape}")

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

pos_weight = torch.tensor(class_weights * POS_WEIGHT_SCALE, dtype=torch.float32).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"pos_weight (scaled x{POS_WEIGHT_SCALE}): {pos_weight.cpu().numpy().round(3)}")

optimizer    = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * NUM_EPOCHS
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)
print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {WARMUP_STEPS}")


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = loss_fn(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (step + 1) % 20 == 0:
            print(f"  Step {step+1}/{len(loader)} | loss: {loss.item():.4f}")

    return total_loss / len(loader)


def eval_epoch(model, loader, loss_fn, device, threshold=THRESHOLD):
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_labels  = []
    all_probs   = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits
            loss    = loss_fn(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int()

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.vstack(all_probs)
    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels).astype(int)

    print(f"  Sigmoid probs — min: {all_probs.min():.4f}  max: {all_probs.max():.4f}  "
          f"mean: {all_probs.mean():.4f}  pos_preds: {all_preds.sum()}")

    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    avg_loss  = total_loss / len(loader)

    return avg_loss, micro_f1, macro_f1, all_preds, all_labels


In [ ]:
history        = []
best_macro_f1  = -1.0  

print("=" * 70)
print("Starting training...")
print(f"Epochs: {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LEARNING_RATE}  |  Threshold: {THRESHOLD}")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, micro_f1, macro_f1, _, _ = eval_epoch(model, test_loader, loss_fn, device)

    row = {
        "epoch"     : epoch,
        "train_loss": round(train_loss, 4),
        "val_loss"  : round(val_loss,   4),
        "micro_f1"  : round(micro_f1,   4),
        "macro_f1"  : round(macro_f1,   4),
    }
    history.append(row)

    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")
    print(f"  Micro F1   : {micro_f1:.4f}")
    print(f"  Macro F1   : {macro_f1:.4f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✓ Checkpoint saved (Macro F1: {macro_f1:.4f})")

print("\n" + "=" * 70)
print(f"Training complete. Best Macro F1: {best_macro_f1:.4f}")
print("=" * 70)

hist_df = pd.DataFrame(history)
print("\nTraining history:")
print(hist_df.to_string(index=False))


In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
_, micro_f1, macro_f1, all_preds, all_labels = eval_epoch(model, test_loader, loss_fn, device)

per_label_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
h_loss       = hamming_loss(all_labels, all_preds)

print("\n" + "=" * 60)
print("DISTILBERT RESULTS")
print("=" * 60)
print(f"  Micro F1      : {micro_f1:.4f}")
print(f"  Macro F1      : {macro_f1:.4f}")
print(f"  Hamming Loss  : {h_loss:.4f}  (lower is better)")
print()

BASELINE = {
    "All-or-nothing thinking"      : 0.0690,
    "Overgeneralization"           : 0.1351,
    "Mental filter"                : 0.2593,
    "Should statements"            : 0.2162,
    "Labeling"                     : 0.0727,
    "Personalization"              : 0.1967,
    "Catastrophising/Magnification": 0.2400,
    "Emotional Reasoning"          : 0.1633,
    "Mind Reading"                 : 0.3390,
    "Fortune-telling"              : 0.2051,
}

print(f"  {'Label':<35} {'Baseline':>8} {'DistilBERT':>10} {'Delta':>7}")
print(f"  {'-'*35} {'-'*8} {'-'*10} {'-'*7}")
for label, f1, baseline in zip(DISTORTION_LABELS, per_label_f1, BASELINE.values()):
    delta = f1 - baseline
    sign  = "+" if delta >= 0 else ""
    flag  = "  ✓" if delta > 0 else "  ✗"
    print(f"  {label:<35} {baseline:>8.4f} {f1:>10.4f} {sign}{delta:>6.4f}{flag}")

print("=" * 60)
print(f"  Baseline Micro F1 : 0.2095  →  DistilBERT: {micro_f1:.4f}")
print(f"  Baseline Macro F1 : 0.1896  →  DistilBERT: {macro_f1:.4f}")

results = {
    "model"        : "DistilBERT fine-tuned",
    "micro_f1"     : round(float(micro_f1), 4),
    "macro_f1"     : round(float(macro_f1), 4),
    "hamming_loss" : round(float(h_loss),   4),
    "per_label_f1" : {
        label: round(float(score), 4)
        for label, score in zip(DISTORTION_LABELS, per_label_f1)
    },
    "training_history": history
}

with open("distilbert_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: distilbert_results.json")

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

with open(os.path.join(MODEL_DIR, "distilbert_results.json"), "w") as f:
    json.dump(results, f, indent=2)

label_config = {
    "distortion_labels": DISTORTION_LABELS,
    "num_labels"        : NUM_LABELS,
    "threshold"         : THRESHOLD,
    "id2label"          : {str(i): l for i, l in enumerate(DISTORTION_LABELS)},
    "label2id"          : {l: i for i, l in enumerate(DISTORTION_LABELS)}
}
with open(os.path.join(MODEL_DIR, "label_config.json"), "w") as f:
    json.dump(label_config, f, indent=2)

print(f"Model saved to: {MODEL_DIR}/")
print("Contents:")
for fname in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, fname))
    print(f"  {fname:<40} {size/1024/1024:.2f} MB")

zip_name = "distilbert_cognitive_distortion"
shutil.make_archive(zip_name, "zip", MODEL_DIR)
zip_size = os.path.getsize(zip_name + ".zip") / 1024 / 1024
print(f"\nZipped: {zip_name}.zip  ({zip_size:.1f} MB)")

from google.colab import files
files.download(zip_name + ".zip")
files.download("distilbert_results.json")
print("Download started.")